In [3]:



# Install required libraries
!pip -q install -U transformers gradio

# Import required libraries
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load pretrained model and tokenizer
model_name = "distilgpt2"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading pretrained model...")
model = AutoModelForCausalLM.from_pretrained(model_name)

# Configure the model
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id
model.eval()

print("Model loaded successfully!")
print("Device:", "GPU" if torch.cuda.is_available() else "CPU")


# Generate chatbot response
def chatbot_response(message, history):
    # Prepare conversation history
    prompt = ""

    for item in history[-6:]:
        if isinstance(item, dict):
            role = item.get("role", "")
            content = item.get("content", "")

            if role in ("user", "assistant") and isinstance(content, str):
                prompt += f"{role.capitalize()}: {content}\n"

    # Add the current user prompt
    prompt += f"User: {message}\nAssistant:"

    # Tokenize input
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # Generate response
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.15
        )

    # Decode generated response
    input_length = inputs["input_ids"].shape[1]
    response = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    ).strip()

    # Remove unwanted extra conversation turns
    for marker in ["\nUser:", "\nAssistant:"]:
        response = response.split(marker)[0].strip()

    return response or "Please try asking your question differently."


# Create the Gradio chatbot interface
demo = gr.ChatInterface(
    fn=chatbot_response,
    title="Simple LLM Chatbot",
    description=(
        "Experiment 10: Building a Simple LLM Chatbot using Gradio. "
        "Ask questions and test text generation."
    ),
    examples=[
        "What is artificial intelligence?",
        "Explain machine learning simply.",
        "What is a neural network?",
        "Why is Python popular?",
        "Tell me about deep learning."
    ]
)

# Launch the chatbot
print("Launching chatbot...")
demo.launch(share=True)


Loading tokenizer...
Loading pretrained model...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Model loaded successfully!
Device: CPU
Launching chatbot...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5ca9be6dc3a49bbd9a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
